![banner](https://github.com/hello-robot/stretch_mujoco/raw/main/docs/images/stretch_mujoco.png)

<h1><center>Getting Started Tutorial  <a href="https://colab.research.google.com/github/hello-robot/stretch_mujoco/blob/main/docs/getting_started.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" width="140" align="center"/></a></center></h1>

This notebook provides an introduction to Stretch's [**MuJoCo** simulation](https://github.com/hello-robot/stretch_mujoco/). You can run this notebook with either **CPU** or **GPU** instance.


## Install

In [76]:
# # Set up Stretch Mujoco repo
# # %rm -rf ./stretch_mujoco/
# # # !git clone https://github.com/hello-robot/stretch_mujoco --recurse-submodules
# # !git clone https://github.com/hello-robot/stretch_mujoco --depth 1
# # %cd ./stretch_mujoco/
# # %pip install -e ".[jupyter]"

# # Check if we can use GPU rendering
# import os
# import subprocess
# try:
#     subprocess.run('nvidia-smi')
#     USE_GPU=True
# except:
#     USE_GPU=False

# # Setup rendering
# if USE_GPU:
#     # Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
#     # This is usually installed as part of an Nvidia driver package, but the Colab
#     # kernel doesn't install its driver via APT, and as a result the ICD is missing.
#     # (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
#     NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
#     if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
#         with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
#             f.write("""{
#       "file_format_version" : "1.0.0",
#       "ICD" : {
#           "library_path" : "libEGL_nvidia.so.0"
#       }
#   }""")

#     # Configure MuJoCo to use the EGL rendering backend (requires GPU)
#     print('Setting environment variable to use GPU rendering:')
#     %env MUJOCO_GL=egl
# else:
#     # Required for OSMesa OpenGL driver
#     !apt-get update
#     !apt-get install -y libosmesa6-dev libgl1-mesa-glx libglfw3

#     print('Setting environment variable to use CPU rendering:')
#     %env MUJOCO_GL=osmesa

# # Other imports and helper functions
# import time
# import pprint
# import itertools
# import numpy as np

# # Graphics and plotting.
# !command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)
# !pip install -q mediapy
# import mediapy as media
# import matplotlib.pyplot as plt

# # More legible printing from numpy.
# np.set_printoptions(precision=3, suppress=True, linewidth=100)

After installing, we need robocasa set up

In [77]:
# python -m pip install -e ".[robocasa]"
# python -m pip install -e third_party/robosuite
# python -m pip install -e third_party/robocasa

If you get an error like

(stretch_mujoco_v310) orrijoa@orrijoa-IdeaPad-Gaming-3-15ACH6:~/projects/stretch_mujoco_jupyter/stretch_mujoco$ python -m pip install -e third_party/robosuite
Obtaining file:///home/orrijoa/projects/stretch_mujoco_jupyter/stretch_mujoco/third_party/robosuite
ERROR: file:///home/orrijoa/projects/stretch_mujoco_jupyter/stretch_mujoco/third_party/robosuite does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.

From the root of your stretch_mujoco repo, run:
git submodule update --init


In [78]:
# python third_party/robosuite/robosuite/scripts/setup_macros.py
# python third_party/robocasa/robocasa/scripts/setup_macros.py
# python third_party/robocasa/robocasa/scripts/download_kitchen_assets.py

## Basics

The `StretchMujocoSimulator` class is used to:

 - Start/stop the simulation
 - Read camera imagery
 - Read lidar scans
 - Read joint states
 - Position control the robot's ranged joints
 - Velocity control the robot's mobile base

### Prerequisites to run to stream cameras in each windows

In [79]:
# Viewer on NVIDIA dGPU (windowed)
import os
os.environ.pop("MUJOCO_GL", None)              # ensure not headless
os.environ["__NV_PRIME_RENDER_OFFLOAD"] = "1"
os.environ["__GLX_VENDOR_LIBRARY_NAME"] = "nvidia"
os.environ["__VK_LAYER_NV_optimus"] = "NVIDIA_only"

# If you want headless instead, comment the above and use:
# import os, shutil
# os.environ["MUJOCO_GL"] = "egl" if shutil.which("nvidia-smi") else "osmesa"

In [80]:
import time
import threading
from pprint import pprint

import numpy as np
import matplotlib.pyplot as plt
import cv2

# More legible printing from numpy.
np.set_printoptions(precision=3, suppress=True, linewidth=100)

# Optional dependency
try:
    import mediapy as media
    HAS_MEDIAPY = True
except ImportError:
    HAS_MEDIAPY = False
    print("mediapy not installed. Install with: python -m pip install mediapy")
    
# Project imports last (after env is set!)
from stretch_mujoco import StretchMujocoSimulator
from stretch_mujoco.enums.stretch_sensors import StretchSensors
from stretch_mujoco.enums.stretch_cameras import StretchCameras
from stretch_mujoco.enums.actuators import Actuators

### Camera & Streaming Set Up

In [81]:
# keep track of which windows we've created
_created_windows = set()

def show_camera_feeds_sync(sim, print_fps=False, init_size=(640, 480)):
    """
    Pull camera data from the simulator and display it using OpenCV.
    Windows are resizable (drag corners).
    """
    camera_data = sim.pull_camera_data()

    if print_fps:
        s = sim.pull_status()
        print(f"Physics fps: {s.fps}. Camera FPS: {camera_data.fps}. {s.sim_to_real_time_ratio_msg}")

    for cam_enum, pixels in camera_data.get_all(use_depth_color_map=True).items():
        if pixels is None:
            continue

        name = cam_enum.name  # window title

        # create a resizable window once per camera
        if name not in _created_windows:
            cv2.namedWindow(name, cv2.WINDOW_NORMAL)       # <-- resizable
            cv2.resizeWindow(name, *init_size)             # initial size
            # optional: keep aspect ratio when resizing
            try:
                cv2.setWindowProperty(name, cv2.WND_PROP_ASPECT_RATIO, cv2.WINDOW_KEEPRATIO)
            except Exception:
                pass
            _created_windows.add(name)

        cv2.imshow(name, pixels)

    # process window events (needed for resizing to take effect)
    cv2.waitKey(1)
    
_stream_evt = None
_stream_thread = None

def start_stream_thread(sim, print_fps=False, target_hz=20):
    """Run camera streaming in the background using your show_camera_feeds_sync()."""
    global _stream_evt, _stream_thread
    if _stream_thread and _stream_thread.is_alive():
        print("Stream already running.")
        return

    _stream_evt = threading.Event()

    def _worker():
        try:
            dt = 1.0 / max(1, target_hz)
            while not _stream_evt.is_set() and sim.is_running():
                show_camera_feeds_sync(sim, print_fps)
                # sim.step()                       # advance physics
                time.sleep(dt)                   # throttle display FPS a bit
        except Exception as e:
            print("stream thread ended:", type(e).__name__, e)
        finally:
            # Close any OpenCV windows cleanly
            try:
                cv2.destroyAllWindows()
            except: 
                pass

    _stream_thread = threading.Thread(target=_worker, daemon=True)
    _stream_thread.start()
    print("Stream thread started.")

def stop_stream_thread():
    """Stop the background stream cleanly (call before sim.stop())."""
    global _stream_evt, _stream_thread
    if _stream_evt:
        _stream_evt.set()
    if _stream_thread:
        _stream_thread.join(timeout=2.0)
    try:
        cv2.waitKey(1)
        cv2.destroyAllWindows()
    except:
        pass
    _stream_evt = None
    _stream_thread = None
    print("Stream thread stopped.")


### Testing for Robocasa Set up

In [82]:
# Core libs
import mujoco
import numpy as np

# RoboCasa generator
try:
    from stretch_mujoco.robocasa_gen import model_generation_wizard
    print("Found model_generation_wizard()")
except Exception as e:
    print("Could not import model_generation_wizard:", e)

# robosuite / robocasa sanity
import robosuite
print("robosuite version:", getattr(robosuite, "__version__", "unknown"))

import robocasa
print("robocasa version:", getattr(robocasa, "__version__", "unknown"))

# Show robosuite macro backend if present
try:
    from robosuite import macros as RS_MACROS
    print("robosuite MUJOCO_GL:", getattr(RS_MACROS, "MUJOCO_GL", "not set"))
except Exception as e:
    print("robosuite macros import issue:", e)


Found model_generation_wizard()
robosuite version: 1.5.1
robocasa version: 0.2.0
robosuite MUJOCO_GL: not set


### Set Up Mujoco Env

In [83]:
model = xml = objects_info = None

try:
    # Non interactive example. Adjust task/layout/style later if you want.
    # These names are common defaults; if they ever change, the except block will let you pick via wizard.
    model, xml, objects_info = model_generation_wizard(
        task="PnPCounterToCab",
        layout=0,
        style=0
    )
    print("Generated RoboCasa model non-interactively.")
except TypeError:
    # Some versions use only the interactive wizard
    print("Non-interactive args not supported. Opening interactive wizard...")
    model, xml, objects_info = model_generation_wizard()
except FileNotFoundError as e:
    print("Asset missing:", e)
    print("Re-run the RoboCasa asset downloader script and try again.")
    raise
except Exception as e:
    print("RoboCasa scene generation failed:", e)
    raise

print("Model OK:", isinstance(model, mujoco.MjModel))
if xml:
    print("XML length:", len(xml))
if objects_info is not None:
    # objects_info is usually a dict from the generator
    print("Objects info keys:", list(objects_info)[:5])


[robosuite INFO] Loading controller configuration from: /home/orrijoa/projects/stretch_mujoco_fork/third_party/robosuite/robosuite/controllers/config/robots/default_pandaomron.json (composite_controller_factory.py:121)


Initializing environment...
Initial observation keys: odict_keys(['robot0_joint_pos_cos', 'robot0_joint_pos_sin', 'robot0_joint_vel', 'robot0_eef_pos', 'robot0_eef_quat', 'robot0_eef_quat_site', 'robot0_gripper_qpos', 'robot0_gripper_qvel', 'robot0_base_pos', 'robot0_base_quat', 'robot0_base_to_eef_pos', 'robot0_base_to_eef_quat', 'robot0_base_to_eef_quat_site', 'apple0_pos', 'apple0_quat', 'apple0_to_robot0_eef_pos', 'apple0_to_robot0_eef_quat', 'avocado0_pos', 'avocado0_quat', 'avocado0_to_robot0_eef_pos', 'avocado0_to_robot0_eef_quat', 'banana0_pos', 'banana0_quat', 'banana0_to_robot0_eef_pos', 'banana0_to_robot0_eef_quat', 'robot0_proprio-state', 'object-state'])
env.object_cfgs after override:
0: name=apple0      cat=apple       model=/home/orrijoa/projects/stretch_mujoco_fork/third_party/robocasa/robocasa/models/assets/objects/objaverse/apple/apple_0/model.xml
1: name=avocado0    cat=avocado     model=/home/orrijoa/projects/stretch_mujoco_fork/third_party/robocasa/robocasa/models

In [84]:
# Pretty-print objects and their initial placements
for body_name, info in objects_info.items():
    print(f"{body_name:30s}  cat={info['cat']:12s}  pos={info['pos']}  quat={info['quat']}")

apple0_main                     cat=apple         pos=(0.35473998355491365, -0.47056294639251434, 0.96314560905)  quat=[1.    0.    0.    0.013]
avocado0_main                   cat=avocado       pos=(0.65143802530092, -0.10624368650618976, 0.9697274932800001)  quat=[1.    0.    0.    0.012]
banana0_main                    cat=banana        pos=(1.4518501728443802, -0.1484498524500852, 1.4454152821139667)  quat=[ 0.937  0.     0.    -0.349]


### Actual Testing with Tele Ops

In [85]:
# sim = StretchMujocoSimulator(cameras_to_use=StretchCameras.all())

# cameras_to_use = [StretchCameras.cam_nav_rgb]
# cameras_to_use = StretchCameras.all()
cameras_to_use = []

sim = StretchMujocoSimulator(model=model, cameras_to_use=cameras_to_use)
sim.start(show_viewer_ui=False, headless=False)

Starting Stretch Mujoco Simulator...
Still waiting to connect to the Mujoco Simulatior.
Still waiting to connect to the Mujoco Simulatior.
Still waiting to connect to the Mujoco Simulatior.
Still waiting to connect to the Mujoco Simulatior.
Still waiting to connect to the Mujoco Simulatior.
Still waiting to connect to the Mujoco Simulatior.
Using the Mujoco Passive Viewer. Note: UI thread and camera rendering is capped to 30.0Hz to increase performance. You can set this rate using the `camera_rate` arugment.
Still waiting to connect to the Mujoco Simulatior.
The Mujoco Simulatior is connected.
Timeout: Joint lift is still moving after 5.0.


### GET OBJ STATE (APPLE)

In [86]:
sim.register_tracked_objects(list(objects_info.keys()))

In [87]:
obj_name = "apple0_main"

In [100]:
obj_state = sim.pull_objects_state([obj_name])
print(obj_state)

{'apple0_main': {'pos': array([ 0.358, -0.47 ,  0.952]), 'quat': array([ 0.999, -0.012,  0.047,  0.013])}}


In [228]:
limits = {
    Actuators.lift: (0.0, 1.1),
    Actuators.arm:  (0.0, 0.52),
    Actuators.gripper: (-0.25, 0.53),
}

# how long we wait between actions.
dt = 0.05
# action scales (start small, tune later)
scales = np.array([0.03, 0.03, 0.02], dtype=np.float32)  # [lift, arm, gripper]

In [229]:
def get_ee_pos(sim):
    T = sim.get_ee_pose()
    return T[:3, 3].astype(float)

def get_obj_pos(sim, obj_name):
    return sim.pull_objects_state()[obj_name]["pos"].astype(float)

def distance_to_object(sim, obj_name):
    ee_pos = get_ee_pos(sim)
    obj_pos = get_obj_pos(sim, obj_name)
    d = float(np.linalg.norm(ee_pos - obj_pos))
    return d, ee_pos, obj_pos

def manual_step(sim, a, obj_name):
    """
    a: array-like shape (3,), values in [-1, 1]
    returns: d, ee_pos, obj_pos
    """
    a = np.asarray(a, dtype=np.float32)
    a = np.clip(a, -1.0, 1.0) # a is just the policy saying “move up/down a bit” (a direction and strength).
    delta = a * scales # scales is shape (3,)

    s = sim.pull_status()
    current_lift = float(s.lift.pos)
    current_arm  = float(s.arm.pos)
    current_grip = float(s.gripper.pos)
    
    lift_low, lift_high = limits[Actuators.lift]
    arm_low,  arm_high  = limits[Actuators.arm]
    grip_low, grip_high = limits[Actuators.gripper]
    
    target_lift = np.clip(current_lift + float(delta[0]), lift_low, lift_high)
    target_arm  = np.clip(current_arm  + float(delta[1]), arm_low,  arm_high)
    target_grip = np.clip(current_grip + float(delta[2]), grip_low, grip_high)

    sim.move_to(Actuators.lift,    target_lift)
    sim.move_to(Actuators.arm,     target_arm)
    sim.move_to(Actuators.gripper, target_grip)

    # fixed control tick
    time.sleep(dt)

    return distance_to_object(sim, obj_name)


In [171]:
distance_to_object(sim, obj_name)

(1.9905265206838072,
 array([ 2.181, -0.832,  0.241]),
 array([ 0.358, -0.47 ,  0.952]))

In [175]:
# move only lift up
for i in range(5):
    d, ee, obj = manual_step(sim, a=[1, 0, 0], obj_name=obj_name)
    s = sim.pull_status()
    print(f"i={i} d={d:.6f} lift={s.lift.pos:.6f} ee_z={ee[2]:.6f}")

i=0 d=1.974046 lift=0.179859 ee_z=0.288160
i=1 d=1.973905 lift=0.180281 ee_z=0.288577
i=2 d=1.973833 lift=0.180573 ee_z=0.288791
i=3 d=1.973643 lift=0.181062 ee_z=0.289352
i=4 d=1.973525 lift=0.181506 ee_z=0.289703


In [179]:
# move only arm forward
for i in range(5):
    d, ee, obj = manual_step(sim, a=[0, 1, 0], obj_name=obj_name)
    s = sim.pull_status()
    print(f"i={i} d={d:.6f} arm={s.arm.pos:.6f} ee_x={ee[0]:.6f} ee_z={ee[2]:.6f}")

i=0 d=1.525452 arm=0.031003 ee_x=1.692185 ee_z=0.308262
i=1 d=1.525729 arm=0.031283 ee_x=1.692485 ee_z=0.308226
i=2 d=1.526103 arm=0.031687 ee_x=1.692890 ee_z=0.308180
i=3 d=1.526592 arm=0.032217 ee_x=1.693420 ee_z=0.308121
i=4 d=1.527302 arm=0.032988 ee_x=1.694191 ee_z=0.308038


In [216]:
# move only gripper (try open)
for i in range(5):
    d, ee, obj = manual_step(sim, a=[0, 0, 1], obj_name=obj_name)
    s = sim.pull_status()
    print(f"i={i} d={d:.6f} gripper={s.gripper.pos:.6f}")


i=0 d=1.579990 gripper=0.035051
i=1 d=1.580013 gripper=0.035051
i=2 d=1.580051 gripper=0.035051
i=3 d=1.580094 gripper=0.035051
i=4 d=1.580125 gripper=0.035051


In [242]:
# move only gripper (try close)
for i in range(5):
    d, ee, obj = manual_step(sim, a=[0, 0, -1], obj_name=obj_name)
    s = sim.pull_status()
    print(f"i={i} d={d:.6f} gripper={s.gripper.pos:.6f}")


i=0 d=1.593959 gripper=-0.010172
i=1 d=1.593991 gripper=-0.011754
i=2 d=1.594036 gripper=-0.014475
i=3 d=1.594083 gripper=-0.017506
i=4 d=1.594131 gripper=-0.020766


In [ ]:
start_stream_thread(sim, print_fps=False, target_hz=20)

### ROBOT MANIPULATION

In [ ]:
# from -2.02 to 0.49 (document) - tested use this value
# from -1.53 to 0.79 (current set up) - correct

sim.move_to(Actuators.head_tilt, 0)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.head_tilt, 0.2)
time.sleep(0.5)

In [ ]:
# from -4.04 to 1.73
sim.move_to(Actuators.head_pan, 1.73)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.head_pan, -0.2)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.base_translate, 0.01)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.base_translate, -1)
time.sleep(0.5)

In [ ]:
# 90 degree ~= 1.57
sim.move_by(Actuators.base_rotate, 0.8)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.base_rotate, -0.01)
time.sleep(0.5)

In [108]:
# from 0 to 1.1 - tested
sim.move_to(Actuators.lift, 1.5)
time.sleep(0.5)

In [151]:
sim.move_by(Actuators.lift, -0.7)
time.sleep(0.5)

In [177]:
# from 0 to 0.13 (current)
# from 0 to 0.52 (document) - correct - tested
sim.move_to(Actuators.arm, 0.0)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.arm, 0.13)
time.sleep(0.5)

In [ ]:
# (-1.39, 4.42) - tested
sim.move_to(Actuators.wrist_yaw, 0)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.wrist_yaw, -0.2)
time.sleep(0.5)

In [ ]:
# (-1.57, 0.56) current - correct - tested
# (-1.57, 0.57) document
sim.move_to(Actuators.wrist_pitch, 0.0)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.wrist_pitch, -0.2)
time.sleep(0.5)

In [ ]:
# (-3.14, 3.14)
sim.move_to(Actuators.wrist_roll, 0)
time.sleep(0.5)

In [ ]:
sim.move_by(Actuators.wrist_roll, -0.2)
time.sleep(0.5)

In [217]:
# (-0.02, 0.04)
# (-0.3,0.55) - tested
sim.move_to(Actuators.gripper, -0.3)
time.sleep(0.5)

In [222]:
sim.move_by(Actuators.gripper, 0.05)
time.sleep(0.5)

In [ ]:
# # Try to force it "very closed"
# sim.move_to(Actuators.gripper, -1.0)
# time.sleep(1)
# print("closed?", sim.pull_status().gripper.pos)

# # Try to force it "very open"
# sim.move_to(Actuators.gripper,  1.0)
# time.sleep(1)
# print("open?", sim.pull_status().gripper.pos)


closed? -0.24973180105830461
open? 0.5312878746843409


### Report current status and limits

In [ ]:
pprint(sim.pull_status())

StatusStretchJoints(time=293.9000000006752,
                    fps=104.47520609985173,
                    sim_to_real_time_ratio_msg='Sim is running 0.209x as fast '
                                               'as realtime',
                    base=BaseStatus(x=1.246528817676964,
                                    y=-0.8108741570298221,
                                    theta=1.5706262409178127,
                                    x_vel=-1.6277763723854874e-08,
                                    theta_vel=8.188364433226527e-08),
                    lift=PositionVelocity(pos=0.13285700041236398,
                                          vel=-1.5404569545683704e-07),
                    arm=PositionVelocity(pos=0.5199826616379475,
                                         vel=1.9862531771636135e-06),
                    head_pan=PositionVelocity(pos=-5.008775231254165e-06,
                                              vel=3.369349949734667e-05),
                    head_tilt=Pos

In [ ]:
pprint(sim.pull_camera_data())



In [ ]:
# radar related
pprint(sim.pull_sensor_data())

In [156]:
print(sim.pull_joint_limits())

{<Actuators.right_wheel_vel: 11>: (0.0, 0.0), <Actuators.left_wheel_vel: 10>: (0.0, 0.0), <Actuators.lift: 4>: (0.0, 1.1), <Actuators.arm: 0>: (0.0, 0.13), <Actuators.wrist_yaw: 7>: (-1.39, 4.42), <Actuators.wrist_pitch: 5>: (-1.57, 0.56), <Actuators.wrist_roll: 6>: (-3.14, 3.14), <Actuators.gripper: 1>: (-0.02, 0.04), <Actuators.gripper_left_finger: 12>: (-0.6, 0.6), <Actuators.gripper_right_finger: 13>: (-0.6, 0.6), <Actuators.head_pan: 2>: (-4.04, 1.73), <Actuators.head_tilt: 3>: (-1.53, 0.79)}


### End the simulation

In [75]:
# # Try to undo any previous manual wrapping if you still have the originals
# try:
#     sys.stdout = _sys_stdout_orig
#     sys.stderr = _sys_stderr_orig
# except NameError:
#     pass

# 1) stop background streaming first
stop_stream_thread()

# 2) give the sim process a short moment to finish its own threads
time.sleep(0.1)

# 3) stop the simulator
if sim.is_running():
    sim.stop()

# 4) as a final sweep, make sure no OpenCV windows remain
try:
    cv2.waitKey(1)
    cv2.destroyAllWindows()
except:
    pass


Stream thread stopped.
Stopping Stretch Mujoco Simulator... simulated runtime= 532.9s
Sending signal to stop the Mujoco process...
Physics Loop has terminated.
Stopping thread 1/1 on the Mujoco Process.
Mujoco viewer has terminated.


Xlib:  extension "NV-GLX" missing on display ":1".


The Mujoco process has ended.
Stopping thread 1/6.
IOPub is not terminating. Make sure to check 'sim.is_running()' in threading loops.
Stopping thread 2/6.
Heartbeat is not terminating. Make sure to check 'sim.is_running()' in threading loops.
Stopping thread 3/6.
Thread-1 (_watch_pipe_fd) is not terminating. Make sure to check 'sim.is_running()' in threading loops.
Stopping thread 4/6.
Thread-2 (_watch_pipe_fd) is not terminating. Make sure to check 'sim.is_running()' in threading loops.
Stopping thread 5/6.
Control is not terminating. Make sure to check 'sim.is_running()' in threading loops.
Stopping thread 6/6.
IPythonHistorySavingThread is not terminating. Make sure to check 'sim.is_running()' in threading loops.
The Stretch Mujoco Simulator has ended. Good-bye!
